# NLP Task 5: Fine-Tuning BERT for POS Tagging & Chunking

"""
Objective:
To fine-tune a transformer model (DistilBERT) for token classification tasks:
1. Part-of-Speech (POS) Tagging
2. Chunking (Phrase Detection)
"""

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

In [4]:
!pip install --upgrade datasets transformers huggingface_hub

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/637.4 kB ? eta -:--:--
   -------------------------------- ------- 524.3/637.4 kB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 637.4/637.4 kB 3.9 MB/s  0:00:00
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.2 MB 5.4 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/10.2 MB 5.7 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/10.2 MB 5.5 MB/s eta 0:00:02
   ---------------- ----------------------- 4.2/10.2 MB 5.4 MB/s eta 0:00:02
   -------------------- ------------------- 5.2/10.2 MB 5.2 MB/s eta 0:00:01
   ------------------------ --------------- 6.3/10.2 MB 5.1 MB/s eta 0:00:01
   ---------------------------- ----------- 7.3/10.2 MB 5.1 MB/s eta 0:00:01
   --------------------------------- ------ 8.7/10.2 MB 5.1 MB/s eta 0:00:01
   -----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [6]:
!pip install transformers datasets seqeval

Defaulting to user installation because normal site-packages is not writeable


In [8]:
!pip install -q transformers datasets seqeval

In [11]:
import requests

def download_file(url, filename):
    response = requests.get(url)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(response.text)

# Download all 3 files
download_file(
    "https://raw.githubusercontent.com/davidsbatista/NER-datasets/master/CONLL2003/train.txt",
    "train.txt"
)

download_file(
    "https://raw.githubusercontent.com/davidsbatista/NER-datasets/master/CONLL2003/valid.txt",
    "valid.txt"
)

download_file(
    "https://raw.githubusercontent.com/davidsbatista/NER-datasets/master/CONLL2003/test.txt",
    "test.txt"
)

print("Download complete!")

Download complete!


In [12]:
import os
print(os.listdir())

['.ipynb_checkpoints', 'dataset.csv', 'GEN AI DAY 1.ipynb', 'GEN AI DAY 2.ipynb', 'GEN AI DAY 3.ipynb', 'test.txt', 'train.txt', 'Untitled.ipynb', 'Untitled1.ipynb', 'valid.txt']


In [15]:
def load_conll(file_path):
    sentences = []
    labels = []

    with open(file_path, "r", encoding="utf-8") as f:
        words = []
        tags = []

        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words = []
                    tags = []
            else:
                splits = line.strip().split()
                words.append(splits[0])
                tags.append(splits[-1])

    return {"tokens": sentences, "labels": labels}

In [16]:
train_data = load_conll("train.txt")
val_data = load_conll("valid.txt")
test_data = load_conll("test.txt")

print(train_data["tokens"][:2])
print(train_data["labels"][:2])

[]
[]


In [18]:
train_data = load_conll("train.txt")
val_data = load_conll("valid.txt")
test_data = load_conll("test.txt")

In [19]:
from datasets import Dataset

train_dataset = Dataset.from_dict(train_data)
val_dataset = Dataset.from_dict(val_data)
test_dataset = Dataset.from_dict(test_data)

In [20]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(label2id[example["labels"][word_idx]])
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [21]:
train_dataset = train_dataset.map(tokenize_and_align_labels)
val_dataset = val_dataset.map(tokenize_and_align_labels)

In [22]:
from transformers import TrainingArguments, Trainer

In [25]:
!pip install accelerate

Defaulting to user installation because normal site-packages is not writeable


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [26]:
pip install "accelerate>=1.1.0"

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [28]:
!pip install -U transformers accelerate torch

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/114.6 MB ? eta -:--:--
   ---------------------------------------- 1.3/114.6 MB 5.8 MB/s eta 0:00:20
    --------------------------------------- 2.1/114.6 MB 4.8 MB/s eta 0:00:24
   - -------------------------------------- 3.1/114.6 MB 4.6 MB/s eta 0:00:25
   - -------------------------------------- 4.2/114.6 MB 4.7 MB/s eta 0:00:24
   - -------------------------------------- 5.2/114.6 MB 4.8 MB/s eta 0:00:23
   -- ------------------------------------- 6.3/114.6 MB 4.8 MB/s eta 0:00:23
   -- ------------------------------------- 7.3/114.6 MB 4.9 MB/s eta 0:00:23
   -- ------------------------------------- 7.9/114.6 MB 4.5 MB/s eta 0:00:24
   --- ------------------------------------ 8.9/114.6 MB 4.6 MB/s eta 0:00:23
   --- ------------------------------------ 10.0/114.6 MB 4.7 MB/s eta 0:00:23

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.25.0 requires torch==2.10.0, but you have torch 2.11.0 which is incompatible.


In [29]:
import accelerate
print(accelerate.__version__)

1.13.0
